# 🤖 Customer Churn: Classification Modeling & Evaluation
**Goal:** Build, benchmark, and evaluate Logistic Regression, Random Forest, and XGBoost models.
**Metrics:** Accuracy, Recall, and ROC-AUC with 5-Fold Stratified Cross-Validation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')

## 1. Load Processed Datasets

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}')

## 2. Define Model Architectures with Class Balancing

In [ ]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f'Minority class scale weight: {pos_weight:.2f}')

models = {
    'Logistic Regression': LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=8, min_samples_split=10, min_samples_leaf=4, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1)
}

## 3. Train Models & 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
records = []

for name, model in models.items():
    print(f'Evaluating {name}...')
    cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    cv_recall = cross_val_score(model, X_train, y_train, cv=cv, scoring='recall').mean()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {'y_pred': y_pred, 'y_prob': y_prob, 'model': model}
    records.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Recall': round(rec, 4),
        'ROC-AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'F1-Score': round(f1, 4),
        '5-Fold CV ROC-AUC': round(cv_auc, 4),
        '5-Fold CV Recall': round(cv_recall, 4)
    })

comparison_df = pd.DataFrame(records)
display(comparison_df)

## 4. Visual Comparison: ROC Curves & Confusion Matrices

In [ ]:
plt.figure(figsize=(8, 6))
colors = {'Logistic Regression': '#1f77b4', 'Random Forest': '#2ca02c', 'XGBoost': '#d62728'}
for name, data in results.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_prob'])
    score = roc_auc_score(y_test, data['y_prob'])
    plt.plot(fpr, tpr, color=colors[name], lw=2.5, label=f'{name} (ROC-AUC = {score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Chance (0.50)')
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.legend(loc='lower right')
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'])
    ax.set_title(f'{name}\nRecall: {recall_score(y_test, data["y_pred"]):.3f}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 6. Feature Importances (XGBoost)

In [ ]:
xgb_model = results['XGBoost']['model']
feat_imp = pd.DataFrame({'Feature': X_train.columns, 'Importance': xgb_model.feature_importances_})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False).head(12)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Top 12 Most Predictive Features (XGBoost)', fontsize=13, fontweight='bold')
plt.show()